## Visualise annotations from MultiByPass140 dataset

In [ ]:
import pickle
from natsort import natsorted
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

# load a sample split from a center
label_path = "./labels/bern/labels_by70_splits/labels/train/1fps_100_0_with_iae.pickle"
with open(label_path, 'rb') as f:
    data = pickle.load(f)

## Plot progression of phase and step

In [ ]:
def plot_progression(surgery_id=None, phase_mapping=None, step_mapping=None):
    [surgery_id] = random.sample(sorted(data), k=1) if surgery_id is None else [surgery_id]
    
    frame_ids = np.array([int(f['Frame_id']) for f in data[surgery_id]])
    phases = np.array([f['Phase_gt'] for f in data[surgery_id]])
    steps = np.array([f['Step_gt'] for f in data[surgery_id]])

    # assign colors for phases and steps
    unique_phases = np.unique(phases)
    phase_cm = plt.cm.tab10(np.linspace(0, 1, len(unique_phases)))
    phase_colors = {phase: phase_cm[i] for i, phase in enumerate(unique_phases)}
    
    unique_steps = np.unique(steps)
    step_cm = plt.cm.tab20(np.linspace(0, 1, len(unique_steps)))
    step_colors = {step: step_cm[i] for i, step in enumerate(unique_steps)}

    fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

    # ------------------ phase progression ------------------
    current_phase = phases[0]
    start_idx = 0
    for i in range(1, len(phases)):
        if phases[i] != current_phase or i == len(phases) - 1:
            end_idx = i if phases[i] != current_phase else i + 1
            ax[0].barh(0, frame_ids[end_idx-1] - frame_ids[start_idx], 
                       left=frame_ids[start_idx], 
                       color=phase_colors[current_phase], 
                       height=1, edgecolor='black', linewidth=0.5)
            start_idx = i
            current_phase = phases[i]

    ax[0].set_xlim(frame_ids[0], frame_ids[-1])
    ax[0].set_ylim(-0.5, 0.5)
    ax[0].set_title(f'Phase progression for Surgery: {surgery_id}')
    ax[0].set_yticks([])
    ax[0].set_xticks([])

    # map the legend to the names
    phase_legend = []
    for p in unique_phases:
        name = phase_mapping.get(int(p), str(p)) if phase_mapping else str(p)
        phase_legend.append(
            plt.Rectangle((0,0),1,1, facecolor=phase_colors[p], label=f'{p}: {name}')
        )
    ax[0].legend(handles=phase_legend, loc='upper center',
                 bbox_to_anchor=(0.5, -0.2), ncol=5, frameon=False)

    # ------------------ step progression ------------------
    current_step = steps[0]
    start_idx = 0
    for i in range(1, len(steps)):
        if steps[i] != current_step or i == len(steps) - 1:
            end_idx = i if steps[i] != current_step else i + 1
            ax[1].barh(0, frame_ids[end_idx-1] - frame_ids[start_idx], 
                       left=frame_ids[start_idx], 
                       color=step_colors[current_step], 
                       height=1, edgecolor='black', linewidth=0.5)
            start_idx = i
            current_step = steps[i]

    ax[1].set_xlim(frame_ids[0], frame_ids[-1])
    ax[1].set_ylim(-0.5, 0.5)
    ax[1].set_title(f'Step progression for Surgery: {surgery_id}')
    ax[1].set_xlabel('Frame progression')
    ax[1].set_yticks([])
    ax[1].set_xticks([])

    # map the legend to the names
    step_legend = []
    for s in unique_steps:
        name = step_mapping.get(int(s), str(s)) if step_mapping else str(s)
        step_legend.append(
            plt.Rectangle((0,0),1,1, facecolor=step_colors[s], label=f'{s}: {name}')
        )
    ax[1].legend(handles=step_legend, loc='upper center',
                 bbox_to_anchor=(0.5, -0.25), ncol=5, frameon=False)
    
    plt.tight_layout()
    plt.show()
    return phases, steps

phase_df = pd.read_csv("./tables/phase.csv", sep=",")
phase_df.columns = phase_df.columns.str.strip()
phase_df["name"] = phase_df["name"].str.strip()
phase_mapping = dict(zip(phase_df["id"], phase_df["name"]))

step_df = pd.read_csv("./tables/step.csv", sep=",")
step_df.columns = step_df.columns.str.strip()
step_df["name"] = step_df["name"].str.strip()
step_mapping = dict(zip(step_df["id"], step_df["name"]))

phases, steps = plot_progression(phase_mapping=phase_mapping, step_mapping=step_mapping)

## Plot the progression of adverse events

In [ ]:
def plot_adverse_event(event_name, surgery_id=None):
    [surgery_id] = random.sample(sorted(data), k=1) if surgery_id is None else [surgery_id]

    frame_ids = np.array([int(f['Frame_id']) for f in data[surgery_id]])
    event_values = np.array([f[event_name] for f in data[surgery_id]])  # expects 0/1

    colors = {0: "lightgrey", 1: "red"}

    fig, ax = plt.subplots(1, 1, figsize=(12, 2))

    current_val = event_values[0]
    start_idx = 0
    for i in range(1, len(event_values)):
        if event_values[i] != current_val or i == len(event_values) - 1:
            end_idx = i if event_values[i] != current_val else i + 1
            ax.barh(0, frame_ids[end_idx-1] - frame_ids[start_idx],
                    left=frame_ids[start_idx],
                    color=colors.get(current_val, "black"),
                    height=1, edgecolor='black', linewidth=0.3)
            start_idx = i
            current_val = event_values[i]

    ax.set_xlim(frame_ids[0], frame_ids[-1])
    ax.set_ylim(-0.5, 0.5)
    ax.set_title(f"{event_name.title()} progression for Surgery: {surgery_id}")
    ax.set_yticks([])
    ax.set_xticks([])

    legend = [
        plt.Rectangle((0,0),1,1, facecolor="red", label="Present"),
        plt.Rectangle((0,0),1,1, facecolor="lightgrey", label="Absent")
    ]
    ax.legend(handles=legend, loc='upper center', bbox_to_anchor=(0.5, -0.25), 
              ncol=2, frameon=False)

    plt.tight_layout()
    plt.show()
    return event_values

# to visualise a particular surgery ID, call the function like :
# plot_adverse_event("Bleeding", surgery_id='BBP01');
# if not provided, it plots a random surgery

plot_adverse_event("Bleeding");
plot_adverse_event("Thermal injury");
plot_adverse_event("Mechanical injury");
plot_adverse_event("Ischemic injury");
plot_adverse_event("Insufficient closure of anastomosis");